# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url
2009-10-01	Paper Title Number 1	Journal 1	This paper is about the number 1. The number 2 is left for future work.	Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).	paper-title-number-1	http://academicpages.github.io/files/paper1.pdf
2010-10-01	Paper Title Number 2	Journal 1	This paper is about the number 2. The number 3 is left for future work.	Your Name, You. (2010). "Paper Title Number 2." <i>Journal 1</i>. 1(2).	paper-title-number-2	http://academicpages.github.io/files/paper2.pdf
2015-10-01	Paper Title Number 3	Journal 1	This paper is about the number 3. The number 4 is left for future work.	Your Name, You. (2015). "Paper Title Number 3." <i>Journal 1</i>. 1(3).	paper-title-number-3	http://academicpages.github.io/files/paper3.pdf

## Import pandas

We are using the very handy pandas library for dataframes.

In [1]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [65]:
publications = pd.read_csv("new_publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url
0,2024-10-01,DTG : Diffusion-based Trajectory Generation fo...,2024 IEEE/RSJ International Conference on Inte...,We present a novel end-to-end diffusion-based ...,"Liang, J., Payandeh, A., Song, D., Xiao, X. an...",dtg,https://arxiv.org/pdf/2403.09900
1,2024-10-01,PoCo: Point Context Cluster for RGBD Indoor Pl...,2024 IEEE/RSJ International Conference on Inte...,We present a novel end-to-end algorithm (PoCo)...,"Liang, J., Deng, Z., Zhou, Z., Ghasemalizadeh,...",poco,https://arxiv.org/pdf/2404.02885
2,2024-10-01,Deep Stochastic Kinematic Models for Probabili...,2024 IEEE/RSJ International Conference on Inte...,"In trajectory forecasting tasks for traffic, f...","Zheng, L., Son, S., Liang, J., Wang, X., Clipp...",motionforcasting,https://arxiv.org/pdf/2406.01431
3,2024-05-13,Mtg: Mapless trajectory generator with travers...,2024 IEEE International Conference on Robotics...,We present a novel learning-based trajectory g...,"Liang, J., Gao, P., Xiao, X., Sathyamoorthy, A...",mtg,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...
4,2023-10-01,Geometric-Preserved Place Recognition for Cros...,2023 IEEE/RSJ International Conference on Inte...,Place recognition plays an important role in m...,"Gao, P., Liang, J., Shen, Y., Son, S. and Lin,...",geometric,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...
5,2023-09-27,GrASPE: Graph based Multimodal Fusion for Robo...,IEEE Robotics and Automation Letters,We present a novel trajectory traversability e...,"Weerakoon, K., Sathyamoorthy, A.J., Liang, J.,...",graspe,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...
6,2023-07-11,MSVIPER,Journal of the Washington Academy of Sciences,We present Multiple Scenario Verifiable Reinfo...,"Roth, A.M., Liang, J., Sriram, R., Tabassi, E....",msviper,https://www.jstor.org/stable/pdf/27281325.pdf?...
7,2023-10-01,CrossLoc3D: Aerial-Ground Cross-Source 3D Plac...,the IEEE/CVF International Conference on Compu...,"We present CROSSLOC3D, a novel 3D place recogn...","Guan, T., Muthuselvam, A., Hoover, M., Wang, X...",crossloc3D,https://openaccess.thecvf.com/content/ICCV2023...
8,2022-12-16,Adaptiveon: Adaptive outdoor local navigation ...,IEEE Robotics and Automation Letters,We present a novel outdoor navigation algorith...,"\n\nLiang, J., Weerakoon, K., Guan, T., Karape...",adaptiveon,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...
9,2022-10-23,Terrapn: Unstructured terrain navigation using...,2022 IEEE/RSJ International Conference on Inte...,"We present TerraPN, a novel method that learns...","Sathyamoorthy, A.J., Weerakoon, K., Guan, T., ...",terrapn,https://ieeexplore.ieee.org/stamp/stamp.jsp?ar...


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [68]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [71]:
import os
for row, item in publications.iterrows():
    print(str(item.pub_date))
    print(item.url_slug)
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_news/" + md_filename, 'w') as f:
        f.write(md)

2024-10-01
dtg
2024-10-01
poco
2024-10-01
motionforcasting
2024-05-13
mtg
2023-10-01
geometric
2023-09-27
graspe
2023-07-11
msviper
2023-10-01
crossloc3D
2022-12-16
adaptiveon
2022-10-23
terrapn
2022-05-30
image-goal
2021-09-27
xain
2021-06-18
ofvo
2021-01-07
crowdsteer
2021-01-06
social_ped
2020-05-31
densecavoid


These files are in the publications directory, one directory below where we're working from.

In [74]:
!ls ../_publications/

2020-05-31-densecavoid.md  2023-07-11-msviper.md
2021-01-06-social_ped.md   2023-09-27-graspe.md
2021-01-07-crowdsteer.md   2023-10-01-crossloc3D.md
2021-06-18-ofvo.md	   2023-10-01-geometric.md
2021-09-27-xain.md	   2024-05-13-mtg.md
2022-05-30-image-goal.md   2024-10-01-dtg.md
2022-10-23-terrapn.md	   2024-10-01-motionforcasting.md
2022-12-16-adaptiveon.md   2024-10-01-poco.md


In [76]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

cat: ../_publications/2009-10-01-paper-title-number-1.md: No such file or directory
